In [4]:
import pandas as pd
from sklearn import metrics, naive_bayes
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score)
import numpy as np
import matplotlib.pyplot as plt

In [7]:
def evaluate_threshold(threshold, probs, y_true, title):
    y_pred = (probs >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"\n=== {title} ===")
    print(f"Threshold: {threshold:.2f}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 score:  {f1:.4f}")
    print("Confusion matrix (rows=true [0,1], cols=pred [0,1]):")
    print(cm)
    print("\nC. report:\n", classification_report(y_true, y_pred, digits=4, zero_division=0))

In [5]:
df = pd.read_csv("/content/Blood Transfusion Service data.csv")
df.head()

,id,Recency,Frequency,Monetary,Time,Class(Target)
0,1,2,50,12500,98,2
1,2,0,13,3250,28,2
2,3,1,16,4000,35,2
3,4,2,20,5000,45,2
4,5,1,24,6000,77,1


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             748 non-null    int64
 1   Recency        748 non-null    int64
 2   Frequency      748 non-null    int64
 3   Monetary       748 non-null    int64
 4   Time           748 non-null    int64
 5   Class(Target)  748 non-null    int64
dtypes: int64(6)
memory usage: 35.2 KB


In [11]:
X = df.drop(columns=['id', 'Class(Target)'])
y = df['Class(Target)'].map({1: 0, 2: 1})


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

probs_test = knn.predict_proba(X_test)[:, 1]
evaluate_threshold(0.50, probs_test, y_test, 'Baseline')
evaluate_threshold(0.25, probs_test, y_test, 'Niži treshold (+25% from 0.25)')
evaluate_threshold(0.75, probs_test, y_test, 'Viši treshold (+75% from 0.75)')


=== Baseline ===
Threshold: 0.50
Accuracy:  0.7600
Precision: 0.5000
Recall:    0.3333
F1 score:  0.4000
Confusion matrix (rows=true [0,1], cols=pred [0,1]):
[[102  12]
 [ 24  12]]

C. report:
               precision    recall  f1-score   support

           0     0.8095    0.8947    0.8500       114
           1     0.5000    0.3333    0.4000        36

    accuracy                         0.7600       150
   macro avg     0.6548    0.6140    0.6250       150
weighted avg     0.7352    0.7600    0.7420       150


=== Niži treshold (+25% from 0.25) ===
Threshold: 0.25
Accuracy:  0.7267
Precision: 0.4468
Recall:    0.5833
F1 score:  0.5060
Confusion matrix (rows=true [0,1], cols=pred [0,1]):
[[88 26]
 [15 21]]

C. report:
               precision    recall  f1-score   support

           0     0.8544    0.7719    0.8111       114
           1     0.4468    0.5833    0.5060        36

    accuracy                         0.7267       150
   macro avg     0.6506    0.6776    0.6585    

In [ ]:
# Zaključak: Niži treshold je najbolja opcija